## Action Responder Prompt

This notebook hopes to:

- Gather trace data for action-responder agent
- Structure and save as mlflow dataset
- Utilize judges to evaluate current agent
- Try with better model to confirm judges are working
- Run GEPA optimzation to improve

In [4]:
import mlflow
from mlflow.tracking import MlflowClient

from summit_sim.settings import settings

mlflow.tracing.enable_notebook_display()

client = MlflowClient()
mlflow.set_tracking_uri(settings.mlflow_tracking_uri)
mlflow.set_experiment(settings.mlflow_experiment_name)
experiment = client.get_experiment_by_name(settings.mlflow_experiment_name)


# Get all traces from last 7 days (adjust as needed)
filter_string = """
tags.graph_type = 'simulation' AND
tags.agent_name = 'action-responder'
"""
traces = client.search_traces(
    locations=[experiment.experiment_id],
    filter_string=filter_string,
    max_results=500,
)

In [5]:
spans = [
    span
    for trace in traces
    for span in client.get_trace(trace.info.trace_id).data.spans
    if span.name == "action_response_agent"
]

dataset = [{"inputs": span.inputs, "outputs": span.outputs} for span in spans]

print(f"Found {len(dataset)} action_response_agent spans")

spans

Found 15 action_response_agent spans


[Span(name='action_response_agent', trace_id='tr-050dbc90371a14aa2798144552bef0ac', span_id='04ea920c5752c0b7', parent_id='5fcf78c00ddf2dd8'),
 Span(name='action_response_agent', trace_id='tr-56157c41f6f1998410899d79a383512a', span_id='5f4d0685bac2d809', parent_id='82e7d4041dde6d47'),
 Span(name='action_response_agent', trace_id='tr-ff0707f0bbf979b73094d729737d239e', span_id='bbcafc0a856227b5', parent_id='4b95947f205419d5'),
 Span(name='action_response_agent', trace_id='tr-eb42280a47e8378e9cacb4afb4497f9d', span_id='8337980ce7dd1ddf', parent_id='7f718eaf9d9497c8'),
 Span(name='action_response_agent', trace_id='tr-0299153d6a3b788848ec9b291f298431', span_id='400dad121ab0fd3d', parent_id='f029835724d1874e'),
 Span(name='action_response_agent', trace_id='tr-449e4c5775d6ccc5d83cb4969f92bfeb', span_id='46058d3a0e4524dc', parent_id='6db36a5d7a963eec'),
 Span(name='action_response_agent', trace_id='tr-b11d9a98ba12ca9924cb0dd98b5c2661', span_id='07bcf2b8fc2f5203', parent_id='abbe602b89e7a571'),

[Trace(trace_id=tr-050dbc90371a14aa2798144552bef0ac), Trace(trace_id=tr-56157c41f6f1998410899d79a383512a), Trace(trace_id=tr-ff0707f0bbf979b73094d729737d239e), Trace(trace_id=tr-eb42280a47e8378e9cacb4afb4497f9d), Trace(trace_id=tr-0299153d6a3b788848ec9b291f298431), Trace(trace_id=tr-449e4c5775d6ccc5d83cb4969f92bfeb), Trace(trace_id=tr-b11d9a98ba12ca9924cb0dd98b5c2661), Trace(trace_id=tr-baf3ea4846cba9ab365b3a2044a99581), Trace(trace_id=tr-b350d8ff770eee4eeb32dc92226cbfd9), Trace(trace_id=tr-935a11230bc23f5cfc7088cc4ea47034)]

In [3]:
from mlflow.genai.judges import make_judge

from summit_sim.judges.scoring import SCORING_JUDGE_INSTRUCTIONS

JUDGE_MODEL_ENDPOINT = "openrouter:/anthropic/claude-haiku-4.5"


scoring_judge = make_judge(
    name="scoring-judge",
    instructions=SCORING_JUDGE_INSTRUCTIONS,
    model=JUDGE_MODEL_ENDPOINT,
    feedback_value_type=bool,
)

scorers = [scoring_judge]

results = mlflow.genai.evaluate(
    data=dataset,
    scorers=scorers,
)

2026/03/29 20:17:08 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.


Evaluating:   0%|          | 0/15 [Elapsed: 00:00, Remaining: ?] 

20:17:10 - LiteLLM:INFO: utils.py:3995 - 
LiteLLM completion() model= anthropic/claude-haiku-4.5; provider = openrouter
2026-03-29 20:17:10 [INFO] LiteLLM: 
LiteLLM completion() model= anthropic/claude-haiku-4.5; provider = openrouter
20:17:10 - LiteLLM:INFO: utils.py:3995 - 
LiteLLM completion() model= anthropic/claude-haiku-4.5; provider = openrouter
2026-03-29 20:17:10 [INFO] LiteLLM: 
LiteLLM completion() model= anthropic/claude-haiku-4.5; provider = openrouter
20:17:10 - LiteLLM:INFO: utils.py:3995 - 
LiteLLM completion() model= anthropic/claude-haiku-4.5; provider = openrouter
2026-03-29 20:17:10 [INFO] LiteLLM: 
LiteLLM completion() model= anthropic/claude-haiku-4.5; provider = openrouter
20:17:10 - LiteLLM:INFO: utils.py:3995 - 
LiteLLM completion() model= anthropic/claude-haiku-4.5; provider = openrouter
2026-03-29 20:17:10 [INFO] LiteLLM: 
LiteLLM completion() model= anthropic/claude-haiku-4.5; provider = openrouter
20:17:10 - LiteLLM:INFO: utils.py:3995 - 
LiteLLM completion